# RAG Chatbot - Legal/Professional Version

## What This Does

This notebook creates a chatbot that answers questions about your documents using **HuggingFace Inference API**.

**Workshop Scenario:**
> You are a legal consultant helping clients going through divorce who ran a business together. Wife Lilly invested $50,000 into the business, but it has since gone into difficulties and husband/ex-husband David has been unable to repay Lilly.

**You Need:**
- A free HuggingFace API token (~300 requests/hour)
- Internet connection

**Cost:** 100% FREE

**Supported Document Formats:**
- **PDF** files from Google Drive
- **DOCX** files (Word documents) from Google Drive
- **Google Docs** (automatically exported)

**Features:**
- Pre-loaded documents (no setup required)
- Source citations (see which documents were used)
- Conversation memory (follow-up questions work)
- 30-second timeout protection

---

**Steps:** 9 total | **Time:** ~5 minutes to set up

---
## Step 1: Install Libraries

This installs the required tools. Takes about 30 seconds.

In [ ]:
# Install all required packages
!pip install -q chromadb gradio pypdf sentence-transformers huggingface_hub gdown python-docx

print("✅ All libraries installed successfully!")
print("\n📄 Supported document types:")
print("   • PDF files")
print("   • DOCX files (Word documents)")
print("   • Google Docs (via public link)")
print("\nℹ️  Note: You may see dependency warnings - these are non-critical.")

---
## Step 2: Load Libraries

Load the tools we just installed.

In [ ]:
import os
import time
import asyncio
import requests
import gradio as gr
import gdown
from huggingface_hub import InferenceClient
from pypdf import PdfReader
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print("📄 Ready to process: PDF, DOCX, Google Docs")

---
## Step 3: Download Documents

This downloads the pre-configured documents for the workshop.

**Supported formats:**
- **PDF** files from Google Drive
- **DOCX** files from Google Drive  
- **Google Docs** (automatically exported as text)

**No setup required** - documents are loaded automatically from public links.

In [ ]:
# ============================================================================
# DOCUMENT SOURCES - Pre-configured for workshop
# ============================================================================
# 
# SUPPORTED FORMATS:
#   1. PDF files:    ("filename.pdf", "GOOGLE_DRIVE_FILE_ID", "pdf")
#   2. DOCX files:   ("filename.docx", "GOOGLE_DRIVE_FILE_ID", "docx")
#   3. Google Docs:  ("filename.txt", "GOOGLE_DOC_ID", "gdoc")
#
# HOW TO GET IDs:
#   PDF/DOCX: https://drive.google.com/file/d/FILE_ID_HERE/view
#   Google Doc: https://docs.google.com/document/d/DOC_ID_HERE/edit
#
# ============================================================================

DOCUMENTS = [
    # PDF files from Google Drive
    ("Affidavit_Template.pdf", "1z6fCNZEQloRI_uS6W_DxGJh6sSbBiJUA", "pdf"),
    
    # Google Docs - Legal Documents
    ("Legal_Doc_01.txt", "1NgxBqZoGsNrRAf58xGKm1eIUZvaWDbV8d5fqNBU3qJc", "gdoc"),
    ("Legal_Doc_02.txt", "1mlKiOYJBMBLJMiL2KTi1tfnEMGyGa0h1ygGnvFwb4pI", "gdoc"),
    ("Legal_Doc_03.txt", "16lIisNhgb51M16Ze9PTejz7fzuyOSAQqRiw7UvkbMCQ", "gdoc"),
    ("Legal_Doc_04.txt", "1RkttAgtCNIF65Ezws4ajEJn44_3Siz_2TBirATzdKkc", "gdoc"),
    ("Legal_Doc_05.txt", "1jgedgCeChA7ZOoSyjF8yIbNjU1CdFZFFrl_eqoBDUTU", "gdoc"),
    ("Legal_Doc_06.txt", "1JuU0-ea4y2ts_dZzue5zOt4zgQekusUtg0Se39FIWPM", "gdoc"),
    ("Legal_Doc_07.txt", "1MKgM3z0XJIuYnDC-V1ECQyN9uyrCviKY9t_GvhCJ4ew", "gdoc"),
    ("Legal_Doc_08.txt", "1orR_Y1mqhFCPdB2JIJRZ7WPAOM5kBal_WTl7tMzL5Eo", "gdoc"),
    ("Legal_Doc_09.txt", "1zU9ISyRZWwnujnb5CDPoUk_59qgtXstC0byNExQY6MM", "gdoc"),
    ("Legal_Doc_10.txt", "18v7qVGJg61LUgJ79ptrJUhu07mseWTFuFg1pPfJ5m7Y", "gdoc"),
    ("Legal_Doc_11.txt", "1PfykIupI2QI3b3b6vYPbYFGGNiSnK2Xmdv1xFqtCxlA", "gdoc"),
    ("Legal_Doc_12.txt", "1oTMFxSrfNRmFkaou1VWDaOZuJVgoMqT_bMahQaUfc-Q", "gdoc"),
    ("Legal_Doc_13.txt", "126vyT5g-m8Mb_CtT6jc0iCVqTNsjPCvsPAkBANts-sg", "gdoc"),
    ("Legal_Doc_14.txt", "1xFzbe8n7yQJnq6U0EhSOdqTD__iTgALwI09MAALym_Y", "gdoc"),
    ("Legal_Doc_15.txt", "1GfuxGd-7ypt7IQ26vSm5Lt70xN9orEC1lECldpYPlZQ", "gdoc"),
    ("Legal_Doc_16.txt", "15iudUOsY0h_F2fm4elF-rAD7BGkifO_Plb9jXj4Emzk", "gdoc"),
    ("Legal_Doc_17.txt", "1O7XBh9zKip88Gg7WD_htpXt31BIS0Rnb6ELUd4uUlDI", "gdoc"),
    ("Legal_Doc_18.txt", "1SuNGdwIVo4lMp3a3yZ41xnztlS8IT4mrXmiAV2IrbmE", "gdoc"),
    ("Legal_Doc_19.txt", "1LaOTzKfIm1YkfCoGeqR47ltITr5JUuujNbzXXdadrt8", "gdoc"),
    ("Legal_Doc_20.txt", "1tShiwDjp6bnyO0DsD7ASai4TBQ3oMZfxLI3wuS6IuEk", "gdoc"),
    ("Legal_Doc_21.txt", "1fU_dakiYCwYLndLFCS9HjISZNUQyUEMmCrojpTTGa7o", "gdoc"),
    ("Legal_Doc_22.txt", "10cYrGxOlGwwdFDDpjwq1zVCOEYdj9USYG2asgLzTqtM", "gdoc"),
    ("Legal_Doc_23.txt", "1rf_8_0mtZJyHIQRCsvAm3efbbshbtQTjEACvD8_ccA8", "gdoc"),
    ("Legal_Doc_24.txt", "1b0_Oyg3TYdEysn4eISJYt2KDwbnfu1IDaK8nVNpgiT0", "gdoc"),
    ("Legal_Doc_25.txt", "1tDJu_h0Rekdq-na-TXywe_eqLZT-zgqONIIxNpenMyA", "gdoc"),
]

# ============================================================================
# DOWNLOAD DOCUMENTS
# ============================================================================

DOCS_DIR = "/content/legal_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

print("📥 Downloading documents...\n")
DOCUMENT_PATHS = []

for filename, file_id, doc_type in DOCUMENTS:
    output_path = os.path.join(DOCS_DIR, filename)
    print(f"Downloading: {filename} ({doc_type.upper()})")
    
    try:
        if doc_type == "pdf" or doc_type == "docx":
            # Download from Google Drive
            url = f"https://drive.google.com/uc?id={file_id}"
            gdown.download(url, output_path, quiet=True)
            
        elif doc_type == "gdoc":
            # Export Google Doc as plain text
            url = f"https://docs.google.com/document/d/{file_id}/export?format=txt"
            response = requests.get(url)
            if response.status_code == 200:
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(response.text)
            else:
                print(f"  ❌ Failed to export Google Doc (status {response.status_code})")
                continue
        
        if os.path.exists(output_path):
            DOCUMENT_PATHS.append((output_path, doc_type))
            print(f"  ✅ Downloaded successfully")
        else:
            print(f"  ❌ Download failed")
            
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")

print(f"\n✅ Downloaded {len(DOCUMENT_PATHS)} document(s)")
print(f"📁 Location: {DOCS_DIR}")

if len(DOCUMENT_PATHS) == 0:
    print("\n⚠️  WARNING: No documents downloaded!")
    print("Check: Are the links set to 'Anyone with the link can view'?")

---
## Step 4: Configuration - ONLY API TOKEN REQUIRED! ✏️

**⚠️ You only need to add your HuggingFace token below**

### Get Your FREE HuggingFace Token:
1. Go to: https://huggingface.co/
2. Sign up for free account (email + password)
3. Go to: https://huggingface.co/settings/tokens
4. Click "Create new token"
5. Name it: "Legal RAG Chatbot"
6. Type: **"Fine-grained"** with **"Make calls to Inference Providers"** enabled
7. Click "Generate"
8. Copy the token (starts with `hf_`)
9. Paste it below

**Free Tier:** ~300 requests per hour

In [ ]:
# ============================================================================
#                    RAG CHATBOT - CONFIGURATION
# ============================================================================

# ============================================================================
# SECTION 1: API CREDENTIALS (REQUIRED) ✏️
# ============================================================================

HUGGINGFACE_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"  # ✏️ REPLACE THIS (starts with hf_)

# ============================================================================
# SECTION 2: MODEL SETTINGS
# ============================================================================

# Qwen2.5 - Proven to work on HuggingFace free inference API
MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"  # ✅ Large, high quality, free tier

# Alternative models to try if this doesn't work:
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # Smaller, faster
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # Good alternative

TEMPERATURE = 0.7       # Creativity (0.0 = focused, 1.0 = creative)
MAX_OUTPUT_TOKENS = 300 # Maximum response length

# ============================================================================
# SECTION 3: PERSONA - Divorce & Business Dispute Legal Consultant
# ============================================================================

PERSONA_NAME = "Legal Consultant"

PERSONA_DESCRIPTION = """
You are an experienced legal consultant specializing in divorce proceedings and business disputes.

SCENARIO CONTEXT:
You are helping clients going through divorce who previously ran a business together.
- Wife (Lilly) invested $50,000 into the business
- The business has since gone into financial difficulties
- Husband/ex-husband (David) has been unable to repay Lilly's investment
- Your role is to help find legal resolutions and remedies

COMMUNICATION STYLE:
- Speak in a clear, professional, and empathetic manner
- Use precise legal terminology when appropriate, but explain terms for clarity
- Reference specific sections from the legal documents to support your advice
- Acknowledge the emotional difficulty of divorce while maintaining professional objectivity
- Present options and potential remedies, explaining pros and cons
- Be careful to note when information is incomplete or when professional legal advice should be sought

FOCUS AREAS:
- Division of marital assets and business interests
- Recovery of investment funds
- Affidavit preparation and requirements
- Legal remedies available to the investing spouse
- Documentation requirements for court proceedings
"""

# ============================================================================
# SECTION 4: RETRIEVAL SETTINGS
# ============================================================================

NUM_RETRIEVED_DOCS = 7  # How many document pieces to search
CHUNK_SIZE = 1000       # Characters per chunk
OVERLAP = 200           # Overlap between chunks

# ============================================================================
# SECTION 5: CONVERSATION SETTINGS
# ============================================================================

CONVERSATION_MEMORY = 3  # How many previous exchanges to remember
SHOW_SOURCES = True      # Show which documents were used
DEBUG_MEMORY = False     # Set True to see conversation history sent to API

# ============================================================================
# SECTION 6: STARTER QUESTIONS - Divorce & Investment Scenario
# ============================================================================

STARTER_QUESTIONS = [
    "What legal options does Lilly have to recover her $50,000 investment?",
    "How should Lilly document her investment in the business for court?",
    "What should be included in an affidavit for this case?",
    "Can Lilly claim the investment as a marital asset in the divorce?",
    "What evidence would strengthen Lilly's case for repayment?",
]

# ============================================================================
# INITIALIZATION
# ============================================================================
client = InferenceClient(token=HUGGINGFACE_TOKEN)

print("✅ Configuration complete!")
print("="*60)
print(f"📋 Persona: {PERSONA_NAME}")
print(f"🤖 Model: {MODEL_NAME}")
print(f"📄 Documents loaded: {len(DOCUMENT_PATHS)}")
print(f"🧠 Conversation memory: {CONVERSATION_MEMORY} exchanges")
print(f"📚 Show sources: {'ON ✅' if SHOW_SOURCES else 'OFF'}")
print("="*60)
print("\n📌 Scenario: Divorce & Business Investment Dispute")
print("   Lilly ($50K investment) vs David (unable to repay)")

---
## Step 5: Test API Connection

**Run this first!** This checks if your API token works.

✅ If successful: Continue to next step  
❌ If failed: Check your API token and try again

In [ ]:
print("🧪 Testing HuggingFace API connection...")
print("=" * 60)

try:
    test_response = client.chat_completion(
        messages=[
            {"role": "system", "content": "You are a helpful legal assistant."},
            {"role": "user", "content": "Say 'Hello! API is working!' in a professional style."}
        ],
        model=MODEL_NAME,
        max_tokens=50,
        temperature=0.7
    )
    
    response_text = test_response.choices[0].message.content
    
    print("✅ SUCCESS! HuggingFace API is working!")
    print(f"\nTest Response: {response_text}")
    print("\n" + "=" * 60)
    print("✅ You can proceed with the rest of the notebook!")
    
except Exception as e:
    error_str = str(e).lower()
    print(f"❌ API TEST FAILED!")
    print(f"Error: {str(e)[:200]}")
    print("\n" + "=" * 60)
    print("⚠️  STOP! Fix this issue before proceeding:")
    
    if "503" in str(e) or "loading" in error_str:
        print("  🔄 Model is loading (can take 20-30 seconds)")
        print("  💡 SOLUTION: Wait 30 seconds and run this cell again")
    elif "model" in error_str and ("not found" in error_str or "does not exist" in error_str or "not_found" in error_str):
        print("  🤖 Model not available or requires license")
        print("  💡 SOLUTION:")
        print("     1. The current model may require license acceptance on HuggingFace")
        print("     2. Try changing MODEL_NAME in Step 4 to:")
        print("        MODEL_NAME = \"microsoft/Phi-3.5-mini-instruct\"")
        print("     3. Re-run Step 4, then try this test again")
    elif "401" in str(e) or "403" in str(e) or ("invalid" in error_str and "token" in error_str):
        print("  🔑 Token issue detected")
        print("  💡 SOLUTION:")
        print("     1. Go to https://huggingface.co/settings/tokens")
        print("     2. Create 'Fine-grained' token")
        print("     3. Enable 'Make calls to Inference Providers'")
        print("     4. Copy new token to Step 4 configuration")
    else:
        print("  1. Check your API token is correct (starts with hf_)")
        print("  2. Check your internet connection")
        print("  3. Make sure token has 'Inference' permissions")

---
## Step 6: Read Documents

This reads all documents (PDF, DOCX, Google Docs) and splits them into searchable pieces.

**Time:** ~30 seconds

In [ ]:
def extract_text_from_pdf(file_path):
    """Extract text from a PDF file."""
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"❌ Error reading PDF: {str(e)}")
        return ""

def extract_text_from_docx(file_path):
    """Extract text from a DOCX file."""
    try:
        doc = DocxDocument(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        # Also extract text from tables
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    text += cell.text + " "
                text += "\n"
        return text
    except Exception as e:
        print(f"❌ Error reading DOCX: {str(e)}")
        return ""

def extract_text_from_txt(file_path):
    """Extract text from a plain text file (Google Docs export)."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"❌ Error reading text file: {str(e)}")
        return ""

def extract_text(file_path, doc_type):
    """Extract text based on document type."""
    if doc_type == "pdf":
        return extract_text_from_pdf(file_path)
    elif doc_type == "docx":
        return extract_text_from_docx(file_path)
    elif doc_type == "gdoc":
        return extract_text_from_txt(file_path)
    else:
        print(f"⚠️ Unknown document type: {doc_type}")
        return ""

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Split text into small pieces (chunks) for better searching."""
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        if chunk.strip():
            chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

# Process all documents
print("📖 Reading documents...\n")
all_chunks = []
metadata = []

for file_path, doc_type in DOCUMENT_PATHS:
    filename = os.path.basename(file_path)
    doc_type_label = {"pdf": "PDF", "docx": "DOCX", "gdoc": "Google Doc"}
    print(f"Processing: {filename} ({doc_type_label.get(doc_type, doc_type)})")
    
    if not os.path.exists(file_path):
        print(f"  ⚠️ File not found")
        continue
    
    text = extract_text(file_path, doc_type)
    
    if text:
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        
        for chunk_idx, chunk in enumerate(chunks):
            metadata.append({
                "source": filename,
                "doc_type": doc_type,
                "chunk_id": chunk_idx,
                "total_chunks": len(chunks)
            })
        
        print(f"  ✅ Created {len(chunks)} searchable pieces")
    else:
        print(f"  ⚠️ No text found")

print(f"\n✅ Done!")
print(f"📊 Total searchable pieces: {len(all_chunks)}")

if len(all_chunks) == 0:
    print("\n⚠️  WARNING: No text found in documents!")

---
## Step 7: Create Search Database

This creates a searchable database from the legal documents.

**Time:** ~1 minute

In [ ]:
print("🔧 Creating vector database...")
print("=" * 60)

# Step 1: Initialize embedding model
print("\n1️⃣ Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("   ✅ Embedding model loaded")

# Step 2: Create embeddings
print("\n2️⃣ Creating embeddings for document chunks...")
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
print(f"   ✅ Created {len(embeddings)} embeddings")

# Step 3: Initialize ChromaDB
print("\n3️⃣ Initializing database...")
chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# Step 4: Create collection
print("\n4️⃣ Creating document collection...")
collection_name = "legal_documents"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "Legal documents for RAG chatbot"}
)

# Step 5: Add documents
print("\n5️⃣ Adding documents to database...")
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_end = min(i + batch_size, len(all_chunks))
    collection.add(
        documents=all_chunks[i:batch_end],
        embeddings=embeddings[i:batch_end].tolist(),
        metadatas=metadata[i:batch_end],
        ids=[f"chunk_{j}" for j in range(i, batch_end)]
    )

print("\n" + "=" * 60)
print("✅ Vector database created successfully!")
print(f"📊 Total chunks in database: {collection.count()}")
print("🔍 Ready for semantic search!")
print("=" * 60)

---
## Step 8: Setup Question Answering

This prepares the chatbot to answer questions about the legal documents.

In [ ]:
# Store which documents were used for the last answer
last_sources_used = []

def retrieve_relevant_context(query, n_results=NUM_RETRIEVED_DOCS):
    """Find relevant pieces from documents based on the question."""
    global last_sources_used
    try:
        query_embedding = embedding_model.encode([query])
        
        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=min(n_results, collection.count())
        )
        
        documents = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        
        # Track which documents were used
        last_sources_used = []
        if metadatas:
            seen_sources = set()
            for meta in metadatas[:3]:
                source_name = meta.get('source', 'Unknown')
                if source_name not in seen_sources:
                    last_sources_used.append(source_name)
                    seen_sources.add(source_name)
        
        return documents
    except Exception as e:
        print(f"Error searching: {str(e)}")
        last_sources_used = []
        return []

def generate_response_sync(question, chat_history=None):
    """Get answer from AI using relevant document pieces + conversation memory."""
    context_docs = retrieve_relevant_context(question)
    
    if context_docs:
        context_docs = context_docs[:3]
        context = "\n\n".join(context_docs)
        context = context[:3000]
    else:
        context = "No relevant documents found."
    
    messages = [
        {
            "role": "system",
            "content": f"""{PERSONA_DESCRIPTION}

ANSWERING INSTRUCTIONS:
Use the information below from the legal documents to answer the question.

CONTEXT FROM DOCUMENTS:
{context}

GUIDELINES:
- Ground your answer in the specific facts and details from the documents
- Reference specific sections or clauses when relevant
- If the documents don't fully answer the question, acknowledge limitations
- Suggest what additional information or documents might be helpful
- Be clear, professional, and empathetic"""
        }
    ]
    
    # Add conversation history
    if chat_history and CONVERSATION_MEMORY > 0:
        recent_history = chat_history[-(CONVERSATION_MEMORY):]
        for user_msg, bot_msg in recent_history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})
    
    messages.append({"role": "user", "content": question})
    
    if DEBUG_MEMORY:
        print("\n" + "="*80)
        print("🧠 DEBUG: CONVERSATION MEMORY")
        print(f"📊 Sending {len(messages)} messages to API")
        print("="*80 + "\n")
    
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL_NAME,
            max_tokens=MAX_OUTPUT_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content
        
    except Exception as e:
        error_str = str(e).lower()
        if "503" in str(e) or "loading" in error_str:
            return "⏳ Model is loading... Please wait 20-30 seconds and try again."
        elif "429" in str(e) or "rate limit" in error_str:
            return "⚠️ Rate limit reached. Wait 10-15 minutes."
        else:
            raise e

async def generate_response_async(question, chat_history=None, timeout_seconds=30):
    """Wrapper with 30-second timeout."""
    try:
        response_text = await asyncio.wait_for(
            asyncio.to_thread(generate_response_sync, question, chat_history),
            timeout=timeout_seconds
        )
        return response_text
    except asyncio.TimeoutError:
        return "⏱️ **Timeout** - Took too long. Try a simpler question."
    except Exception as e:
        return f"❌ **Error** - {str(e)[:100]}"

print("✅ Question answering system ready!")
print(f"🤖 Model: {MODEL_NAME}")
print(f"🧠 Memory: {CONVERSATION_MEMORY} exchanges")
print("⏱️  Response time: 5-15 seconds")
if SHOW_SOURCES:
    print("📚 Source citations enabled")

---
## Step 9: Launch Chat Interface

This launches the legal consultation chatbot!

**⚠️ IMPORTANT:** 
- Copy the `https://xxxxx.gradio.live` link
- Open it in a **NEW browser tab** (not in Colab)

**Scenario Reminder:**
> Help Lilly recover her $50,000 investment from the business she ran with David before their divorce.

In [ ]:
async def chat_interface(message, history):
    """Handle chat messages with conversation memory and source citations."""
    response = await generate_response_async(message, history)
    
    if SHOW_SOURCES and last_sources_used:
        response += "\n\n---\n**📚 Sources:**\n"
        for i, source in enumerate(last_sources_used, 1):
            response += f"{i}. {source}\n"
    
    return response

# Create chat interface
demo = gr.ChatInterface(
    fn=chat_interface,
    title="⚖️ Legal Consultant - Divorce & Business Dispute",
    description="""**Scenario:** Help Lilly recover her $50,000 investment from the business she ran with David before their divorce.
    
    🧠 Conversation memory enabled | 📚 Source citations on | ⏱️ Response: 5-15 seconds
    """,
    examples=STARTER_QUESTIONS,
)

# Launch
print("=" * 80)
print("⚖️  LAUNCHING LEGAL CONSULTATION CHATBOT")
print("=" * 80)
print("\n📌 SCENARIO: Divorce & Business Investment Dispute")
print("   • Lilly invested $50,000 in the business")
print("   • Business has gone into difficulties")
print("   • David (ex-husband) unable to repay")
print("   • Goal: Find legal remedies for Lilly")
print("\n⚠️  IMPORTANT: Use the PUBLIC LINK below (not Colab interface)\n")
print("👇 COPY THIS LINK AND OPEN IN NEW TAB:\n")

demo.launch(
    share=True,
    inline=False,
    debug=True
)

print("\n" + "=" * 80)
print("✅ Legal consultation chatbot is live!")
print("=" * 80)